# 2단계: B2B 영업 승패 모델 1차 비교

전처리 노트북의 동일한 13개 범주형 입력으로 여섯 모델을 비교한다. 모델마다 필요한 인코딩만 적용하되 정답, 마스킹, Group CV, 평가 지표는 동일하게 유지한다.

1. Train 313개 원본 행의 마스킹 변형 10세트를 학습에 사용한다.
2. 동일한 13개 입력 조합과 모든 마스킹 변형을 한 그룹으로 묶어 `StratifiedGroupKFold(5)`로 평가한다.
3. 각 모델의 최적 조합은 CV Brier Score가 가장 낮은 값으로 정한다.
4. Test 마스킹 10세트에서는 Brier, Log Loss, AUC, Accuracy, Precision, Recall, F1, FP·FN을 함께 확인한다.
5. 분류 임계값은 모든 모델에서 `0.5`로 고정한다.

| 모델 | 입력 처리 | 포함 이유 |
|---|---|---|
| Dummy | 원본 범주 무시 | 평균 승률 기준선 |
| LogisticRegression | 모델 내부 원핫 | 작은 데이터의 선형 확률 기준 |
| MultinomialNB | 모델 내부 원핫 | 개별 범주 빈도 기준 |
| ExtraTrees | 모델 내부 원핫 | 비선형 배깅 모델 |
| CatBoost | 원본 범주형 | 순서형 타깃 통계 기반 부스팅 |
| TabICL | 원본 범주형 | 사전학습 표형 파운데이션 모델 |

## 0. 실행 환경

```bash
uv sync --project backend/notebooks --locked
uv run --project backend/notebooks --locked jupyter lab backend/notebooks/deal_model_phase2.ipynb
```

원본 CSV 위치가 기본값과 다르면 `SALESLUV_B2B_DATA_PATH` 환경변수로 지정한다. TabICL은 첫 실행에서 공식 체크포인트를 내려받을 수 있다.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

# 저장소 루트, backend 폴더, notebooks 폴더 어디에서 실행해도 전처리 파일을 찾는다.
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

assert preprocessing_notebook.exists(), f"전처리 노트북이 없습니다: {preprocessing_notebook}"
ipython = get_ipython()
assert ipython is not None, "이 파일은 Jupyter에서 실행해야 합니다."
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = globals()["X_train_raw"]
y_train = globals()["y_train"]
train_group_ids = globals()["train_group_ids"]
X_test_raw_sets = globals()["X_test_raw_sets"]
y_test = globals()["y_test"]
MODEL_FEATURE_NAMES = globals()["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = globals()["CATEGORY_VALUES"]

RANDOM_STATE = 1
CLASSIFICATION_THRESHOLD = 0.5
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"전처리 노트북: {preprocessing_notebook}")
print(f"Train: {X_train_raw.shape}, 동일 입력 그룹: {len(np.unique(train_group_ids))}")
print(f"Test: {len(X_test_raw_sets)}세트 × {len(y_test)}행")
print(f"공통 임계값: {CLASSIFICATION_THRESHOLD}")

데이터 전처리 검증을 통과했습니다.
전처리 노트북: backend/notebooks/deal_data_preprocessing.ipynb
Train: (3130, 13), 동일 입력 그룹: 138
Test: 10세트 × 135행
공통 임계값: 0.5


### 해석

- 모든 모델은 동일한 3,130개 학습 행과 138개 입력 그룹을 사용한다.
- 공통 입력은 `Unknown`을 포함한 원본 범주형 13개다. 원핫이 필요한 모델만 자신의 파이프라인에서 변환한다.
- Test 10세트는 파라미터 선택에 사용하지 않고 마스킹 조합에 따른 성능 변동을 확인할 때만 사용한다.

In [2]:
# 모델 비교 전에 전처리 결과와 그룹 분할 조건을 확인한다.
assert X_train_raw.shape == (3130, 13)
assert len(y_train) == 3130
assert len(np.unique(train_group_ids)) == 138
assert len(X_test_raw_sets) == 10
assert all(test_data.shape == (135, 13) for test_data in X_test_raw_sets.values())
assert len(y_test) == 135
assert all(str(dtype) == "category" for dtype in X_train_raw.dtypes)

cv_splits = list(cv5.split(X_train_raw, y_train, groups=train_group_ids))
for train_index, valid_index in cv_splits:
    assert set(train_group_ids[train_index]).isdisjoint(set(train_group_ids[valid_index]))

data_for_modeling = pd.DataFrame(
    {
        "항목": [
            "Train 원본 행 수",
            "Train 동일 입력 그룹 수",
            "Train 마스킹 포함 행 수",
            "공통 범주형 컬럼 수",
            "모델 내부 원핫 컬럼 수",
            "Test 마스킹 세트 수",
            "Test 세트당 행 수",
            "분류 임계값",
        ],
        "값": [313, 138, len(X_train_raw), 13, 39, len(X_test_raw_sets), len(y_test), 0.5],
    }
)
display(data_for_modeling)

,항목,값
0,Train 원본 행 수,313.0
1,Train 동일 입력 그룹 수,138.0
2,Train 마스킹 포함 행 수,3130.0
3,공통 범주형 컬럼 수,13.0
4,모델 내부 원핫 컬럼 수,39.0
5,Test 마스킹 세트 수,10.0
6,Test 세트당 행 수,135.0
7,분류 임계값,0.5


### 해석

- 한 원본의 10개 마스킹 변형과 같은 13개 입력을 가진 반복 행은 학습 Fold와 검증 Fold에 동시에 들어가지 않는다.
- 39개 원핫 컬럼은 LR·NB·ExtraTrees 파이프라인 안에서만 생성된다. 다른 모델은 13개 범주형 컬럼을 직접 사용한다.

## 1. 공통 모델 생성·평가 과정

In [3]:
CV_SCORING = {
    "brier": "neg_brier_score",
    "logloss": "neg_log_loss",
    "auc": "roc_auc",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}
TEST_METRIC_COLUMNS = (
    "brier",
    "logloss",
    "auc",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "fpr",
    "tn",
    "fp",
    "fn",
    "tp",
)


def make_one_hot_model(classifier):
    '''13개 고정 범주를 원핫 인코딩한 뒤 전달하는 모델 파이프라인을 만든다.'''
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


def positive_class_probability(estimator, X):
    '''분류 확률에서 Won=1 열만 반환한다.'''
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


def evaluate_test_sets(estimator) -> pd.DataFrame:
    '''학습이 끝난 한 모델을 Test 마스킹 10세트에서 같은 방식으로 평가한다.'''
    rows = []
    for set_name, X_test in X_test_raw_sets.items():
        probability = positive_class_probability(estimator, X_test)
        prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
        specificity = tn / (tn + fp)
        rows.append(
            {
                "test_set": set_name,
                "brier": brier_score_loss(y_test, probability),
                "logloss": log_loss(y_test, probability, labels=[0, 1]),
                "auc": roc_auc_score(y_test, probability),
                "accuracy": accuracy_score(y_test, prediction),
                "precision": precision_score(y_test, prediction, zero_division=0),
                "recall": recall_score(y_test, prediction, zero_division=0),
                "f1": f1_score(y_test, prediction, zero_division=0),
                "specificity": specificity,
                "fpr": 1 - specificity,
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )
    return pd.DataFrame(rows)


def summarize_test_results(test_results: pd.DataFrame) -> pd.DataFrame:
    '''Test 10세트 결과의 평균·표준편차·최솟값·최댓값을 반환한다.'''
    summary = test_results[list(TEST_METRIC_COLUMNS)].agg(["mean", "std", "min", "max"]).T
    summary.index.name = "metric"
    return summary


def search_result_table(search) -> pd.DataFrame:
    '''모든 파라미터 후보를 CV Brier가 낮은 순서로 정리한다.'''
    return (
        pd.DataFrame(
            {
                "params": search.cv_results_["params"],
                "cv_brier": -search.cv_results_["mean_test_brier"],
                "cv_brier_std": search.cv_results_["std_test_brier"],
                "cv_logloss": -search.cv_results_["mean_test_logloss"],
                "cv_auc": search.cv_results_["mean_test_auc"],
                "cv_accuracy": search.cv_results_["mean_test_accuracy"],
                "cv_precision": search.cv_results_["mean_test_precision"],
                "cv_recall": search.cv_results_["mean_test_recall"],
                "cv_f1": search.cv_results_["mean_test_f1"],
                "mean_fit_time": search.cv_results_["mean_fit_time"],
            }
        )
        .sort_values("cv_brier", ignore_index=True)
    )


def train_and_evaluate(estimator, parameter_grid, *, n_jobs: int):
    '''Group CV로 파라미터를 고르고 Test 10세트를 각각 평가한다.'''
    search = GridSearchCV(
        estimator=estimator,
        param_grid=parameter_grid,
        scoring=CV_SCORING,
        cv=cv_splits,
        n_jobs=n_jobs,
        refit="brier",
        error_score="raise",
        return_train_score=False,
    )
    search.fit(X_train_raw, y_train, groups=train_group_ids)
    test_results = evaluate_test_sets(search.best_estimator_)
    return search, test_results, summarize_test_results(test_results)


model_searches = {}
test_results_by_model = {}
test_summaries_by_model = {}
model_input_types = {}
print("공통 모델 생성·평가 함수를 준비했습니다.")

공통 모델 생성·평가 함수를 준비했습니다.


### 해석

- 모델별 최적화 기준은 CV Brier로 통일한다. AUC와 Accuracy 등은 같은 후보에서 함께 기록해 전체 성능을 확인한다.
- 원핫 인코더의 허용 범주를 1단계 스키마로 고정했으므로 Fold마다 컬럼 수나 범주 코드가 달라지지 않는다.
- 스키마에 없는 값은 조용히 무시하지 않고 오류로 중단한다.

## 2. Dummy 기준선

입력을 사용하지 않고 Train의 평균 승률만 반환한다. 실제 모델이 학습 가능한 신호를 얻었는지 확인하는 기준이다.

In [4]:
model_dummy = DummyClassifier()
param_dummy = {"strategy": ["prior"]}
search_dummy, test_results_dummy, test_summary_dummy = train_and_evaluate(
    model_dummy, param_dummy, n_jobs=-1
)
model_searches["Dummy"] = search_dummy
test_results_by_model["Dummy"] = test_results_dummy
test_summaries_by_model["Dummy"] = test_summary_dummy
model_input_types["Dummy"] = "입력 무시"
print(f"최적 파라미터: {search_dummy.best_params_}")
print(f"최적 CV Brier: {-search_dummy.best_score_:.6f}")
display(test_summary_dummy.round(6))

최적 파라미터: {'strategy': 'prior'}
최적 CV Brier: 0.249936


,mean,std,min,max
metric,,,,
brier,0.250005,0.0,0.250005,0.250005
logloss,0.693156,0.0,0.693156,0.693156
auc,0.500000,0.0,0.500000,0.500000
accuracy,0.503704,0.0,0.503704,0.503704
precision,0.503704,0.0,0.503704,0.503704
recall,1.000000,0.0,1.000000,1.000000
f1,0.669951,0.0,0.669951,0.669951
specificity,0.000000,0.0,0.000000,0.000000
fpr,1.000000,0.0,1.000000,1.000000


### 해석 기준

Dummy보다 Brier가 낮고 AUC가 0.5보다 높은지 확인한다.

## 3. LogisticRegression

13개 범주를 모델 내부에서 원핫 인코딩한 뒤 규제된 선형 결합으로 Won 확률을 계산한다. 작은 데이터에서 안정적인 기준 모델이다.

In [5]:
model_logistic = make_one_hot_model(
    LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
)
param_logistic = {"classifier__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}
search_logistic, test_results_logistic, test_summary_logistic = train_and_evaluate(
    model_logistic, param_logistic, n_jobs=-1
)
model_searches["LogisticRegression"] = search_logistic
test_results_by_model["LogisticRegression"] = test_results_logistic
test_summaries_by_model["LogisticRegression"] = test_summary_logistic
model_input_types["LogisticRegression"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_logistic.best_params_}")
print(f"최적 CV Brier: {-search_logistic.best_score_:.6f}")
display(search_result_table(search_logistic).round(6))
display(test_summary_logistic.round(6))

최적 파라미터: {'classifier__C': 0.03}
최적 CV Brier: 0.197368


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,{'classifier__C': 0.03},0.197368,0.019469,0.583188,0.768105,0.722824,0.710734,0.807157,0.746305,0.016121
1,{'classifier__C': 0.1},0.199058,0.022492,0.592116,0.761834,0.716532,0.712936,0.786609,0.735414,0.014273
2,{'classifier__C': 0.01},0.201950,0.015498,0.592712,0.769692,0.718355,0.696805,0.821172,0.747328,0.011400
3,{'classifier__C': 0.3},0.202377,0.024694,0.607506,0.755598,0.713274,0.710074,0.780901,0.731693,0.012270
4,{'classifier__C': 1.0},0.206308,0.026254,0.627620,0.748993,0.710734,0.707686,0.779006,0.729861,0.011664
5,{'classifier__C': 3.0},0.209386,0.026679,0.649374,0.745821,0.709485,0.705484,0.779107,0.729669,0.015051
6,{'classifier__C': 10.0},0.211769,0.026853,0.677436,0.744524,0.704690,0.698992,0.775981,0.725419,0.014792


,mean,std,min,max
metric,,,,
brier,0.184573,0.006171,0.175047,0.195141
logloss,0.548631,0.014223,0.528313,0.570347
auc,0.805948,0.021349,0.768218,0.834284
accuracy,0.705926,0.026375,0.666667,0.740741
precision,0.720729,0.040372,0.671642,0.800000
recall,0.683824,0.027072,0.647059,0.735294
f1,0.701011,0.022522,0.666667,0.729927
specificity,0.728358,0.052558,0.671642,0.835821
fpr,0.271642,0.052558,0.164179,0.328358


### 해석 기준

`C`가 작을수록 규제가 강하다. Test 표준편차가 작으면 Unknown 위치 변화에 덜 민감하다.

## 4. MultinomialNB

모델 내부 원핫 범주와 승패의 개별 빈도 관계를 학습한다. 범주 간 복잡한 상호작용 없이 얻을 수 있는 확률 성능을 확인한다.

In [6]:
model_nb = make_one_hot_model(MultinomialNB())
param_nb = {
    "classifier__alpha": [0.1, 0.3, 0.6, 1.0, 2.0, 3.0, 5.0],
    "classifier__fit_prior": [True, False],
}
search_nb, test_results_nb, test_summary_nb = train_and_evaluate(
    model_nb, param_nb, n_jobs=-1
)
model_searches["MultinomialNB"] = search_nb
test_results_by_model["MultinomialNB"] = test_results_nb
test_summaries_by_model["MultinomialNB"] = test_summary_nb
model_input_types["MultinomialNB"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_nb.best_params_}")
print(f"최적 CV Brier: {-search_nb.best_score_:.6f}")
display(search_result_table(search_nb).round(6))
display(test_summary_nb.round(6))

최적 파라미터: {'classifier__alpha': 5.0, 'classifier__fit_prior': False}
최적 CV Brier: 0.201593


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.201593,0.033914,0.626694,0.763514,0.719536,0.702459,0.806733,0.745632,0.007117
1,"{'classifier__alpha': 5.0, 'classifier__fit_pr...",0.201692,0.034666,0.626997,0.763514,0.718286,0.699025,0.811210,0.746029,0.007654
2,"{'classifier__alpha': 3.0, 'classifier__fit_pr...",0.202279,0.034591,0.635772,0.762645,0.719546,0.703008,0.805462,0.745365,0.007355
3,"{'classifier__alpha': 3.0, 'classifier__fit_pr...",0.202374,0.035344,0.636048,0.762645,0.718604,0.700382,0.809274,0.745710,0.007474
4,"{'classifier__alpha': 2.0, 'classifier__fit_pr...",0.202804,0.034968,0.643093,0.761923,0.718563,0.701560,0.804817,0.744482,0.007570
5,"{'classifier__alpha': 2.0, 'classifier__fit_pr...",0.202898,0.035721,0.643358,0.761923,0.717928,0.699347,0.808629,0.745046,0.008502
6,"{'classifier__alpha': 1.0, 'classifier__fit_pr...",0.203659,0.035338,0.655918,0.760777,0.716278,0.699845,0.800967,0.741984,0.008184
7,"{'classifier__alpha': 1.0, 'classifier__fit_pr...",0.203753,0.036091,0.656176,0.760777,0.716944,0.698543,0.806714,0.743931,0.007382
8,"{'classifier__alpha': 0.6, 'classifier__fit_pr...",0.204231,0.035432,0.665646,0.759881,0.716278,0.699845,0.800967,0.741984,0.007692
9,"{'classifier__alpha': 0.6, 'classifier__fit_pr...",0.204326,0.036187,0.665905,0.759881,0.716606,0.698297,0.806048,0.743547,0.007825


,mean,std,min,max
metric,,,,
brier,0.181306,0.008525,0.171430,0.197384
logloss,0.545317,0.026600,0.515988,0.585882
auc,0.804280,0.020718,0.768218,0.831321
accuracy,0.714815,0.031475,0.666667,0.770370
precision,0.732334,0.035990,0.682540,0.779661
recall,0.685294,0.041710,0.632353,0.764706
f1,0.707460,0.033059,0.656489,0.770370
specificity,0.744776,0.043057,0.671642,0.805970
fpr,0.255224,0.043057,0.194030,0.328358


### 해석 기준

`alpha`가 클수록 희소 범주의 극단적인 확률을 더 강하게 완화한다.

## 5. ExtraTrees

모델 내부 원핫 결과에서 범주 조합의 비선형 상호작용을 여러 무작위 트리로 학습한다.

In [7]:
model_extratrees = make_one_hot_model(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1)
)
param_extratrees = {
    "classifier__n_estimators": [300, 600],
    "classifier__max_depth": [None, 6, 10, 14],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", 0.5, 1.0],
}
search_extratrees, test_results_extratrees, test_summary_extratrees = train_and_evaluate(
    model_extratrees, param_extratrees, n_jobs=-1
)
model_searches["ExtraTrees"] = search_extratrees
test_results_by_model["ExtraTrees"] = test_results_extratrees
test_summaries_by_model["ExtraTrees"] = test_summary_extratrees
model_input_types["ExtraTrees"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_extratrees.best_params_}")
print(f"최적 CV Brier: {-search_extratrees.best_score_:.6f}")
display(search_result_table(search_extratrees).head(20).round(6))
display(test_summary_extratrees.round(6))

최적 파라미터: {'classifier__max_depth': 6, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 4, 'classifier__n_estimators': 300}
최적 CV Brier: 0.199512


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'classifier__max_depth': 6, 'classifier__max_...",0.199512,0.018998,0.587855,0.766045,0.725253,0.702108,0.820343,0.753487,0.209681
1,"{'classifier__max_depth': 6, 'classifier__max_...",0.199593,0.019164,0.588230,0.766575,0.724874,0.701089,0.821554,0.753426,0.414475
2,"{'classifier__max_depth': 6, 'classifier__max_...",0.199604,0.018953,0.588066,0.765240,0.723286,0.700231,0.819679,0.752062,0.210775
3,"{'classifier__max_depth': 6, 'classifier__max_...",0.199797,0.019009,0.588627,0.764772,0.722980,0.700829,0.815927,0.750956,0.207586
4,"{'classifier__max_depth': 6, 'classifier__max_...",0.199833,0.019182,0.588736,0.765443,0.723891,0.699751,0.822179,0.752888,0.408100
5,"{'classifier__max_depth': 10, 'classifier__max...",0.199840,0.021797,0.592906,0.760528,0.713869,0.700013,0.797096,0.738965,0.272017
6,"{'classifier__max_depth': 6, 'classifier__max_...",0.199938,0.019204,0.589061,0.765313,0.722928,0.699473,0.819677,0.751712,0.414982
7,"{'classifier__max_depth': 10, 'classifier__max...",0.199942,0.021621,0.593498,0.759237,0.716482,0.700842,0.802237,0.742314,0.545355
8,"{'classifier__max_depth': 10, 'classifier__max...",0.200767,0.022293,0.595422,0.757338,0.712611,0.695642,0.801028,0.739795,0.559956
9,"{'classifier__max_depth': 10, 'classifier__max...",0.200830,0.022840,0.595136,0.757777,0.714537,0.697109,0.804154,0.741800,0.276145


,mean,std,min,max
metric,,,,
brier,0.183536,0.004754,0.176028,0.190722
logloss,0.549595,0.011129,0.534117,0.569799
auc,0.817493,0.019488,0.778534,0.838455
accuracy,0.747407,0.019269,0.733333,0.792593
precision,0.706495,0.015673,0.690476,0.738095
recall,0.852941,0.024015,0.823529,0.911765
f1,0.772771,0.017706,0.756757,0.815789
specificity,0.640299,0.021629,0.611940,0.671642
fpr,0.359701,0.021629,0.328358,0.388060


### 해석 기준

제한된 깊이와 큰 잎 크기가 선택되면 마스킹 변형을 외우는 깊은 트리보다 단순한 트리가 유리하다는 뜻이다.

## 6. CatBoost

`Unknown`을 포함한 원본 13개 범주를 직접 입력한다. CatBoost의 순서형 타깃 통계와 범주 조합 학습을 사용하므로 사전 원핫 인코딩은 적용하지 않는다.

In [8]:
model_catboost = CatBoostClassifier(
    cat_features=tuple(MODEL_FEATURE_NAMES),
    loss_function="Logloss",
    verbose=False,
    allow_writing_files=False,
    random_seed=RANDOM_STATE,
    random_strength=1.0,
    thread_count=1,
)
param_catboost = {
    "iterations": [300, 600],
    "depth": [4, 6],
    "learning_rate": [0.03, 0.07],
    "l2_leaf_reg": [3.0, 7.0],
}
search_catboost, test_results_catboost, test_summary_catboost = train_and_evaluate(
    model_catboost, param_catboost, n_jobs=-1
)
model_searches["CatBoost"] = search_catboost
test_results_by_model["CatBoost"] = test_results_catboost
test_summaries_by_model["CatBoost"] = test_summary_catboost
model_input_types["CatBoost"] = "원본 범주형 13개"
print(f"최적 파라미터: {search_catboost.best_params_}")
print(f"최적 CV Brier: {-search_catboost.best_score_:.6f}")
display(search_result_table(search_catboost).head(20).round(6))
display(test_summary_catboost.round(6))

최적 파라미터: {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 7.0, 'learning_rate': 0.03}
최적 CV Brier: 0.199160


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'depth': 4, 'iterations': 300, 'l2_leaf_reg':...",0.199160,0.027578,0.608452,0.764151,0.716124,0.709406,0.785359,0.735671,1.051445
1,"{'depth': 4, 'iterations': 300, 'l2_leaf_reg':...",0.199378,0.025584,0.611283,0.764590,0.716452,0.708878,0.784093,0.735337,1.037875
2,"{'depth': 6, 'iterations': 300, 'l2_leaf_reg':...",0.200235,0.027278,0.617235,0.761966,0.716862,0.710892,0.781670,0.735684,1.753288
3,"{'depth': 6, 'iterations': 300, 'l2_leaf_reg':...",0.201454,0.028424,0.621796,0.761559,0.716126,0.711317,0.780302,0.734727,1.747807
4,"{'depth': 4, 'iterations': 600, 'l2_leaf_reg':...",0.203767,0.027909,0.628694,0.757012,0.714793,0.709528,0.777818,0.732940,2.108671
5,"{'depth': 4, 'iterations': 300, 'l2_leaf_reg':...",0.205894,0.025552,0.636576,0.751175,0.708808,0.697557,0.783545,0.730344,1.040512
6,"{'depth': 4, 'iterations': 300, 'l2_leaf_reg':...",0.206127,0.026278,0.642559,0.753741,0.713723,0.701972,0.788606,0.734943,1.079637
7,"{'depth': 4, 'iterations': 600, 'l2_leaf_reg':...",0.206287,0.025690,0.638653,0.753201,0.706683,0.697362,0.779131,0.727912,2.053340
8,"{'depth': 6, 'iterations': 600, 'l2_leaf_reg':...",0.206333,0.028337,0.640639,0.755052,0.708873,0.702616,0.774893,0.729135,3.733999
9,"{'depth': 6, 'iterations': 600, 'l2_leaf_reg':...",0.207250,0.027589,0.646483,0.752827,0.710093,0.704570,0.771549,0.729093,3.750598


,mean,std,min,max
metric,,,,
brier,0.179712,0.006320,0.169399,0.189559
logloss,0.536287,0.016767,0.509671,0.557035
auc,0.806299,0.018768,0.780070,0.839442
accuracy,0.742222,0.021469,0.711111,0.770370
precision,0.720541,0.011639,0.702703,0.736842
recall,0.797059,0.048428,0.705882,0.852941
f1,0.756381,0.026572,0.711111,0.789116
specificity,0.686567,0.017234,0.656716,0.716418
fpr,0.313433,0.017234,0.283582,0.343284


### 해석 기준

기존 원핫 CatBoost 결과가 아니라 범주형 직접 처리 결과다. 같은 CV에서 다른 모델과 Brier·AUC·Accuracy를 다시 비교한다.

## 7. TabICL

원본 13개 범주형 컬럼을 전달하면 TabICL 내부 인코더가 각 범주를 수치 표현으로 바꾼다. 사전 원핫 39개를 다시 입력하지 않는다.

In [9]:
if torch.backends.mps.is_available():
    tabicl_device = "mps"
elif torch.cuda.is_available():
    tabicl_device = "cuda"
else:
    tabicl_device = "cpu"

model_tabicl = TabICLClassifier(
    n_estimators=8,
    batch_size=8,
    kv_cache=False,
    allow_auto_download=True,
    device=tabicl_device,
    use_fa3="auto",
    offload_mode="auto",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=False,
)
param_tabicl = [
    {
        "norm_methods": [None],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True, False],
        "softmax_temperature": [0.9, 1.1],
    },
    {
        "norm_methods": ["quantile"],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True],
        "softmax_temperature": [0.9, 1.1],
    },
]
search_tabicl, test_results_tabicl, test_summary_tabicl = train_and_evaluate(
    model_tabicl, param_tabicl, n_jobs=1
)
model_searches["TabICL"] = search_tabicl
test_results_by_model["TabICL"] = test_results_tabicl
test_summaries_by_model["TabICL"] = test_summary_tabicl
model_input_types["TabICL"] = "원본 범주형 13개"
print(f"장치: {tabicl_device}")
print(f"최적 파라미터: {search_tabicl.best_params_}")
print(f"최적 CV Brier: {-search_tabicl.best_score_:.6f}")
display(search_result_table(search_tabicl).round(6))
display(test_summary_tabicl.round(6))

장치: mps
최적 파라미터: {'average_logits': True, 'feat_shuffle_method': 'latin', 'norm_methods': 'quantile', 'softmax_temperature': 1.1}
최적 CV Brier: 0.213916


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'average_logits': True, 'feat_shuffle_method'...",0.213916,0.030760,0.689077,0.737885,0.694599,0.695493,0.763160,0.715965,0.199717
1,"{'average_logits': True, 'feat_shuffle_method'...",0.218041,0.033636,0.736716,0.737885,0.694599,0.695493,0.763160,0.715965,0.184524
2,"{'average_logits': False, 'feat_shuffle_method...",0.223492,0.025312,0.739917,0.723075,0.689985,0.681386,0.761601,0.712742,0.188292
3,"{'average_logits': True, 'feat_shuffle_method'...",0.223802,0.025446,0.743732,0.723348,0.689985,0.681386,0.761601,0.712742,0.187171
4,"{'average_logits': False, 'feat_shuffle_method...",0.229160,0.027361,0.808769,0.723039,0.689985,0.681386,0.761601,0.712742,0.187770
5,"{'average_logits': True, 'feat_shuffle_method'...",0.229619,0.027509,0.815405,0.723348,0.689985,0.681386,0.761601,0.712742,0.205141


,mean,std,min,max
metric,,,,
brier,0.177992,0.005323,0.171616,0.186647
logloss,0.536877,0.014871,0.514283,0.562750
auc,0.806629,0.015521,0.784021,0.828797
accuracy,0.730370,0.029876,0.688889,0.770370
precision,0.714184,0.025152,0.666667,0.741379
recall,0.776471,0.068206,0.632353,0.852941
f1,0.742526,0.036429,0.682540,0.786207
specificity,0.683582,0.044942,0.611940,0.776119
fpr,0.316418,0.044942,0.223881,0.388060


### 해석 기준

기존 ML보다 Brier·AUC가 모두 좋아야 체크포인트와 추론 비용을 감수할 근거가 생긴다. 단일 순위가 낮아도 오류 상관이 낮으면 앙상블 후보로 남긴다.

## 8. 전체 모델 비교

In [10]:
comparison_rows = []
for model_name, search in model_searches.items():
    summary = test_summaries_by_model[model_name]
    comparison_rows.append(
        {
            "model": model_name,
            "input": model_input_types[model_name],
            "cv_brier": -search.best_score_,
            "cv_brier_std": search.cv_results_["std_test_brier"][search.best_index_],
            "cv_logloss": -search.cv_results_["mean_test_logloss"][search.best_index_],
            "cv_auc": search.cv_results_["mean_test_auc"][search.best_index_],
            "cv_accuracy": search.cv_results_["mean_test_accuracy"][search.best_index_],
            "cv_precision": search.cv_results_["mean_test_precision"][search.best_index_],
            "cv_recall": search.cv_results_["mean_test_recall"][search.best_index_],
            "cv_f1": search.cv_results_["mean_test_f1"][search.best_index_],
            "test_brier_mean": summary.loc["brier", "mean"],
            "test_brier_std": summary.loc["brier", "std"],
            "test_auc_mean": summary.loc["auc", "mean"],
            "test_accuracy_mean": summary.loc["accuracy", "mean"],
            "test_precision_mean": summary.loc["precision", "mean"],
            "test_recall_mean": summary.loc["recall", "mean"],
            "test_f1_mean": summary.loc["f1", "mean"],
            "test_fp_mean": summary.loc["fp", "mean"],
            "test_fn_mean": summary.loc["fn", "mean"],
            "best_params": search.best_params_,
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values("cv_brier", ignore_index=True)
display(comparison.round(6))
real_models = comparison.loc[comparison["model"] != "Dummy"].reset_index(drop=True)
print(f"CV Brier 기준 1순위 실제 모델: {real_models.iloc[0]['model']}")

,model,input,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,test_brier_mean,test_brier_std,test_auc_mean,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_fp_mean,test_fn_mean,best_params
0,LogisticRegression,모델 내부 원핫 39개,0.197368,0.019469,0.583188,0.768105,0.722824,0.710734,0.807157,0.746305,0.184573,0.006171,0.805948,0.705926,0.720729,0.683824,0.701011,18.2,21.5,{'classifier__C': 0.03}
1,CatBoost,원본 범주형 13개,0.199160,0.027578,0.608452,0.764151,0.716124,0.709406,0.785359,0.735671,0.179712,0.006320,0.806299,0.742222,0.720541,0.797059,0.756381,21.0,13.8,"{'depth': 4, 'iterations': 300, 'l2_leaf_reg':..."
2,ExtraTrees,모델 내부 원핫 39개,0.199512,0.018998,0.587855,0.766045,0.725253,0.702108,0.820343,0.753487,0.183536,0.004754,0.817493,0.747407,0.706495,0.852941,0.772771,24.1,10.0,"{'classifier__max_depth': 6, 'classifier__max_..."
3,MultinomialNB,모델 내부 원핫 39개,0.201593,0.033914,0.626694,0.763514,0.719536,0.702459,0.806733,0.745632,0.181306,0.008525,0.804280,0.714815,0.732334,0.685294,0.707460,17.1,21.4,"{'classifier__alpha': 5.0, 'classifier__fit_pr..."
4,TabICL,원본 범주형 13개,0.213916,0.030760,0.689077,0.737885,0.694599,0.695493,0.763160,0.715965,0.177992,0.005323,0.806629,0.730370,0.714184,0.776471,0.742526,21.2,15.2,"{'average_logits': True, 'feat_shuffle_method'..."
5,Dummy,입력 무시,0.249936,0.000003,0.693019,0.500000,0.507992,0.507992,1.000000,0.673733,0.250005,0.000000,0.500000,0.503704,0.503704,1.000000,0.669951,67.0,0.0,{'strategy': 'prior'}


CV Brier 기준 1순위 실제 모델: LogisticRegression


### 해석 기준

- Brier는 확률 오차, AUC는 순위 능력, Accuracy·Precision·Recall·FP·FN은 임계값 0.5의 분류 결과를 보여 준다.
- 한 지표의 작은 차이만으로 모델을 확정하지 않는다. CV 변동과 Test 마스킹 민감도도 함께 본다.
- 원본 범주형 모델의 결과는 기존 원핫 결과와 별개의 새 기준이며 다음 단계에서 다시 정밀 튜닝한다.

In [11]:
expected_models = {
    "Dummy",
    "LogisticRegression",
    "MultinomialNB",
    "ExtraTrees",
    "CatBoost",
    "TabICL",
}
assert set(model_searches) == expected_models
assert comparison["model"].nunique() == len(expected_models)
assert comparison.drop(columns=["model", "input", "best_params"]).notna().all().all()
assert all(len(results) == 10 for results in test_results_by_model.values())
assert all(
    set(results["test_set"]) == set(X_test_raw_sets)
    for results in test_results_by_model.values()
)
print("2단계 모델 비교 검증을 통과했습니다.")

2단계 모델 비교 검증을 통과했습니다.


### 해석

여섯 모델이 같은 원본 범주형 데이터, Group CV, Test 마스킹 세트에서 비교됐다. 모델 파일 저장과 앙상블은 뒤 단계에서 수행한다.